Scenario: AI-Powered Project Tracker (Task-Based)
1. State Definition
The assistant maintains a notebook-like state for each project:
- task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
- status → The current state of the task (e.g., "in progress", "completed", "blocked").
- log → A history of all updates, including who made them and when.

2. Workflow (Graph of States)
Each project update flows through nodes until the task status is refreshed:
- Input Node
- Team member submits an update (e.g., "Report draft completed").
- State updates: task = "Q1 financial report"
- Processing Node (Groq API)
- Groq interprets the update and assigns a status:
- status = "completed"
- Response Node
- Assistant confirms the update back to the team:
- "Task Q1 financial report marked as completed."
- History Node
- Logs the update:
- log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
from google.colab import userdata
from datetime import datetime
import json

# 1. DEFINE STATE
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    log: List[Dict]


# 2. NODE: PROCESS UPDATE (Groq)
def process_update(state: ProjectState):
    groq_api_key = userdata.get('newapi')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets.")

    prompt = f"""
    You are a project management assistant.

    Task: {state['task']}
    Update: {state['update']}

    Determine the task status.

    Possible statuses:
    - completed
    - in progress
    - blocked

    Return ONLY JSON:
    {{
        "status": "..."
    }}
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        raise Exception(f"Groq API error: {response.text}")

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    try:
        result = json.loads(content)
    except:
        raise ValueError(f"Invalid JSON from model: {content}")

    return {"status": result["status"]}


# 3. NODE: RESPONSE
def generate_response(state: ProjectState):
    print(f"\nTask '{state['task']}' marked as: {state['status'].upper()}")

    return {}


# 4. NODE: LOG HISTORY
def log_update(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    updated_log = state["log"] + [entry]

    return {"log": updated_log}


# 5. NODE: SHOW LOG
def show_log(state: ProjectState):
    print("\nProject Update Log:")

    for i, entry in enumerate(state["log"], 1):
        print(f"{i}. [{entry['timestamp']}]")
        print(f"   Task: {entry['task']}")
        print(f"   Update: {entry['update']}")
        print(f"   Status: {entry['status']}\n")

    return {}


# 6. BUILD GRAPH
graph = StateGraph(ProjectState)

graph.add_node("process", process_update)
graph.add_node("response", generate_response)
graph.add_node("log", log_update)
graph.add_node("show_log", show_log)

# Flow
graph.set_entry_point("process")
graph.add_edge("process", "response")
graph.add_edge("response", "log")
graph.add_edge("log", "show_log")
graph.add_edge("show_log", END)

app = graph.compile()


# 7. RUN LOOP
if __name__ == "__main__":
    task = input("Enter task (e.g., Q1 financial report): ")

    state = {
        "task": task,
        "update": "",
        "status": "",
        "log": []
    }

    while True:
        update = input("\nEnter project update: ")
        state["update"] = update

        state = app.invoke(state)

        cont = input("\nAdd another update? (yes/no): ").lower()
        if cont != "yes":
            print("\nProject tracking ended.")
            break

Enter task (e.g., Q1 financial report): Testing Report

Enter project update: We have successfully completed the beta testing.

Task 'Testing Report' marked as: COMPLETED

Project Update Log:
1. [2026-03-20 07:22:18]
   Task: Testing Report
   Update: We have successfully completed the beta testing.
   Status: completed


Add another update? (yes/no): no

Project tracking ended.
